In [ ]:
%pip install -q otter-grader

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook()

# In-Class Exercise 06: $k$-Means Clustering

**DS701 — Session 5 (Mon Sep 21, 2026)**

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Course-Notes-FA26/blob/main/class_activity_notebooks/06-InClass-Exercise-Clustering/06-InClass-Exercise-Clustering.ipynb)

**Time: ~60 minutes.** Work in groups of 2-3.

Parts marked **(autograded)** are submitted to Gradescope; open-ended parts are graded
for participation. You may use AI assistance, but you must be able to explain and
justify every part of your solution when asked.

**Plan**

| Part | Topic | Grading | Time |
|---|---|---|---|
| 1 | One Lloyd's iteration, by hand | autograded | ~18 min |
| 2 | Run to convergence, compare with `sklearn` | autograded | ~12 min |
| 3 | Choosing $k$: elbow and silhouette | partly autograded | ~15 min |
| 4 | Breaking $k$-means | participation | ~15 min |

Dependencies: `numpy`, `matplotlib`, `scikit-learn`.

## Setup

We use a small synthetic dataset: 300 points in $\mathbb{R}^2$ drawn from 4 fairly
well-separated Gaussian blobs. `y_true` is the blob each point came from — we will
*only* use it at the very end to sanity-check, never to fit. Clustering is
unsupervised.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
# Colab: fetch the autograder tests for this activity.
import os, sys, urllib.request

if "google.colab" in sys.modules:
    os.makedirs("tests", exist_ok=True)
    BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main/class_activity_notebooks/06-InClass-Exercise-Clustering/tests/"
    for _q in ("q1", "q2", "q3"):
        urllib.request.urlretrieve(BASE + _q + ".py", "tests/" + _q + ".py")

In [ ]:
RANDOM_STATE = 701

X, y_true = make_blobs(
    n_samples=300, centers=4, cluster_std=0.70, random_state=RANDOM_STATE
)

print("X.shape =", X.shape)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], s=25, alpha=0.7, c="gray")
plt.title("The data (unlabeled, as the algorithm sees it)")
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# A fixed, deliberately arbitrary initialization. Everyone starts from the same
# centers so results are comparable (and gradeable).
k = 4
rng = np.random.default_rng(RANDOM_STATE)
init_centroids = X[rng.choice(len(X), k, replace=False)]
print("initial centroids:\n", np.round(init_centroids, 3))

## Part 1 (autograded): one Lloyd's iteration, by hand

Recall the algorithm:

1. pick $K$ centers $\{c_1,\dots,c_K\}$,
2. **assign** each point to the cluster with the closest center,
3. **update** each center to the center of mass of its cluster,
4. repeat 2-3 until convergence.

and the objective it is trying to minimize, the Within-Cluster Sum of Squares:

$$\text{WCSS} = \sum_{j=1}^{K}\ \sum_{x \in C_j} \lVert x - c_j \rVert^2 .$$

**Task.** Implement the three building blocks below with `numpy`. Do **not** call
`sklearn` here — the point is to know exactly what the library is doing.

- `assign_clusters(X, centroids)` → integer array of shape `(n,)`, entry $i$ is the
  index of the centroid nearest to `X[i]` (Euclidean distance).
- `update_centroids(X, labels, k)` → array of shape `(k, d)`, row $j$ is the mean of
  the points labeled $j$.
- `compute_wcss(X, labels, centroids)` → a single float, the WCSS.

In [ ]:
def assign_clusters(X, centroids):
    """Return the index of the nearest centroid for each row of X."""
    ...


def update_centroids(X, labels, k):
    """Return the (k, d) array of cluster means."""
    ...


def compute_wcss(X, labels, centroids):
    """Return the within-cluster sum of squared distances to the assigned center."""
    ...

### Part 1b (autograded): run exactly one iteration

Starting from `init_centroids`, do **one** assignment step and **one** update step.
Store:

- `labels_1` — the assignment produced by the first assignment step,
- `centroids_1` — the centers after the first update step,
- `wcss_before` — WCSS of `labels_1` with the **old** centers `init_centroids`,
- `wcss_after` — WCSS of `labels_1` with the **new** centers `centroids_1`.

Then answer, in the markdown cell that follows: *why must `wcss_after` be no larger
than `wcss_before`, no matter what the data is?*

In [ ]:
...

print(f"WCSS before update: {wcss_before:.2f}")
print(f"WCSS after  update: {wcss_after:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, cents, title in [
    (axes[0], init_centroids, f"After assignment (WCSS = {wcss_before:.0f})"),
    (axes[1], centroids_1, f"After update (WCSS = {wcss_after:.0f})"),
]:
    ax.scatter(X[:, 0], X[:, 1], c=labels_1, cmap="viridis", s=25, alpha=0.7)
    ax.scatter(cents[:, 0], cents[:, 1], c="red", marker="x", s=200, linewidths=3)
    ax.set_title(title)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Your answer (why can't WCSS go up?):**

*(write 1-2 sentences here)*

In [ ]:
grader.check("q1")

## Part 2 (autograded): run to convergence and compare with `scikit-learn`

**Task.** Loop your Part 1 functions until the centers stop moving, then check yourself
against the library.

1. Write `my_kmeans(X, init_centroids, k, max_iter=100, tol=1e-8)` returning
   `(centroids, labels, wcss, n_iter)`. Stop when every center moves less than `tol`
   (or after `max_iter` passes). Remember to re-assign after the final update so the
   labels match the returned centers.
2. Fit `sklearn.cluster.KMeans` from the **same** starting point:
   `KMeans(n_clusters=k, init=init_centroids, n_init=1, random_state=RANDOM_STATE)`.
   Store it in `km`.
3. Compare `my_wcss` against `km.inertia_`.

Passing `init=init_centroids, n_init=1` disables `sklearn`'s $k$-means++ and its
multiple restarts — so it runs the *same* algorithm from the *same* place as you.

In [ ]:
def my_kmeans(X, init_centroids, k, max_iter=100, tol=1e-8):
    """Run Lloyd's algorithm to convergence. Returns (centroids, labels, wcss, n_iter)."""
    ...


my_centroids, my_labels, my_wcss, my_iters = my_kmeans(X, init_centroids, k)
print(f"mine:    WCSS = {my_wcss:.4f} after {my_iters} iterations")

km = ...
...
print(f"sklearn: WCSS = {km.inertia_:.4f} after {km.n_iter_} iterations")

In [ ]:
grader.check("q2")

Sanity check the result visually, and (only now, only to look) against the blobs the
data actually came from.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(X[:, 0], X[:, 1], c=my_labels, cmap="viridis", s=25, alpha=0.7)
axes[0].scatter(my_centroids[:, 0], my_centroids[:, 1], c="red", marker="x",
                s=200, linewidths=3)
axes[0].set_title(f"my_kmeans (WCSS = {my_wcss:.0f})")
axes[1].scatter(X[:, 0], X[:, 1], c=y_true, cmap="viridis", s=25, alpha=0.7)
axes[1].set_title("ground-truth blobs (peeking only)")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3 (partly autograded): choosing $k$

In Parts 1 and 2 we were handed $k = 4$. In practice $k$ is a **hyperparameter** and
nobody hands it to us. Two standard ways to pick it:

- **Elbow method** — plot WCSS against $k$ and look for the bend where extra clusters
  stop buying much. WCSS decreases monotonically in $k$ (it hits 0 at $k = n$), so the
  minimum is useless; the *shape* is what carries the information.
- **Silhouette score** — for each point compare its mean distance to its own cluster
  against its mean distance to the nearest other cluster. Averaged over points it lands
  in $[-1, 1]$, higher is better, and unlike WCSS it has an interior maximum.

**Task.**

1. Fill `wcss_by_k` with the inertia of `KMeans(n_clusters=kk, n_init=10,
   random_state=RANDOM_STATE)` for `kk` in `k_range = range(1, 11)`.
2. Fill `sil_by_k` with the silhouette score for `kk` in `sil_range = range(2, 11)`
   (silhouette is undefined for a single cluster).
3. Set `best_k_silhouette` to the $k$ with the highest silhouette score.

In [ ]:
k_range = list(range(1, 11))
sil_range = list(range(2, 11))

...

print("WCSS:      ", [round(v, 1) for v in wcss_by_k])
print("silhouette:", [round(v, 3) for v in sil_by_k])
print("best k by silhouette:", best_k_silhouette)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(k_range, wcss_by_k, "bo-", linewidth=2, markersize=7)
axes[0].set_xlabel("number of clusters k")
axes[0].set_ylabel("WCSS")
axes[0].set_title("Elbow method")
axes[1].plot(sil_range, sil_by_k, "go-", linewidth=2, markersize=7)
axes[1].set_xlabel("number of clusters k")
axes[1].set_ylabel("mean silhouette")
axes[1].set_title("Silhouette score")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Discuss and answer (participation):**

1. Where is the elbow in the WCSS plot? Is it as obvious as the textbook picture?
2. WCSS at $k = 10$ is much lower than at $k = 4$. Why is that not an argument for
   $k = 10$?
3. Suppose the two criteria had disagreed — say the elbow looked like 3 and the
   silhouette peaked at 5. What would you do?

*(your answers here)*

In [ ]:
grader.check("q3")

## Part 4 (open-ended, participation): breaking $k$-means

The blobs above were the friendly case: round, similar size, similar spread. Below is
the same generator followed by a **linear transformation** — the clusters are still
Gaussian, just stretched and rotated (*anisotropic*).

Run the cell, then work through the questions. **Be ready to explain your reasoning when
cold-called** — staff will circulate and ask groups to walk through *why* the failure
happens, not just that it does.

In [ ]:
transformation = np.array([[0.60834549, -0.63667341], [-0.40887718, 0.85253229]])
X_blobs, y_aniso = make_blobs(n_samples=600, random_state=170)
X_aniso = X_blobs @ transformation

km_aniso = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(X_aniso)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(X_aniso[:, 0], X_aniso[:, 1], c=y_aniso, cmap="viridis", s=15, alpha=0.7)
axes[0].set_title("Ground truth")
axes[1].scatter(X_aniso[:, 0], X_aniso[:, 1], c=km_aniso.labels_, cmap="viridis",
                s=15, alpha=0.7)
axes[1].scatter(km_aniso.cluster_centers_[:, 0], km_aniso.cluster_centers_[:, 1],
                c="red", marker="x", s=200, linewidths=3)
axes[1].set_title("k-means, k = 3")
for ax in axes:
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

<!-- BEGIN QUESTION -->

### 4a. Diagnose

$k$-means found 3 clusters and its WCSS is a perfectly good local minimum — yet the
partition is wrong. Write down, in your own words:

- What geometric shape does the boundary between two $k$-means clusters always have,
  and why does that follow directly from the assignment step?
- Which assumption baked into the WCSS objective does this dataset violate?
- Would running more iterations, a better initialization, or a larger `n_init` help
  here? Why or why not?

_Type your answer here, replacing this text._

### 4b. Mitigate

Try at least one fix and report whether it helped. Some directions (pick one, or invent
your own):

- **Rescale / whiten.** Feature scaling changes what "distance" means. Try
  `StandardScaler`, or `PCA(n_components=2, whiten=True)` which additionally decorrelates
  the axes, and re-run $k$-means on the transformed data.
- **Change the model.** `sklearn.mixture.GaussianMixture(n_components=3,
  covariance_type="full")` fits an ellipse per cluster instead of a ball — it is
  $k$-means with the spherical assumption removed.
- **Over-cluster and merge.** Run $k$-means with $k = 9$ and see whether the small cells
  respect the true boundaries well enough to be merged.

Use `adjusted_rand_score(y_aniso, labels)` to score each attempt against the truth (1.0
is a perfect match, 0.0 is chance). Plain $k$-means scores about **0.59** here — beat it.

In [ ]:
from sklearn.metrics import adjusted_rand_score

print("plain k-means:", round(adjusted_rand_score(y_aniso, km_aniso.labels_), 3))

...

**Your 4b write-up:** which fix did you try, what adjusted Rand index did it reach,
and *why* did it help (or not)?

*(replace this text with your answer)*

<!-- END QUESTION -->

---

### 4c. If you have time

Build your own failure case and diagnose it the same way. Options:

- `make_moons(n_samples=300, noise=0.05)` — two interleaved crescents, $k = 2$.
- `make_blobs(..., cluster_std=[1.0, 2.5, 0.5])` — unequal variance.
- `np.vstack([X[y == 0][:500], X[y == 1][:100], X[y == 2][:10]])` — unequal *sizes*.

For each, predict what $k$-means will do **before** you run it, then check. Which of
these can feature scaling fix, and which are baked into the objective and cannot be?

In [ ]:
# Scratch space for 4c.


## Wrap-up

Before you leave, each group should be able to answer:

1. Which step of Lloyd's algorithm can increase WCSS? (Trick question.)
2. Why is a lower WCSS not automatically a better clustering?
3. Name one $k$-means failure mode that feature scaling fixes and one that it does not.

Submit the autograded parts (1, 1b, 2, 3) to Gradescope.